In [ ]:
ADAPTATION_POLICY_DESCRIPTION = """Adaptation policies: Policies designed to increase resilience to climate impacts such as extreme weather events, rising sea levels, or changing resource availability. Examples include flood defense infrastructure, heatwave preparedness 
plans, drought-resilient agricultural practices, and urban greening for cooling. 
Policies were classified as adaptation when their main goal was to manage climate 
risks and vulnerabilities."""

In [2]:
from dotenv import load_dotenv
import os

# Force .env values to replace anything already in the environment
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")

In [3]:
from attachments.dspy import Attachments  


In [4]:
from openai import OpenAI


In [5]:
import os
import dspy

lm = dspy.LM(
    model="openai/gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)


In [ ]:
from docling.document_converter import DocumentConverter
src="../pdfs/2021 Portugal ADCOM_UNFCCC.pdf"
converter= DocumentConverter()



ModuleNotFoundError: No module named 'docling'

In [6]:
documents = converter.convert(src)


2026-01-12 21:22:30,192 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-12 21:22:30,328 - INFO - Going to convert document batch...
2026-01-12 21:22:30,329 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-12 21:22:30,343 - INFO - Loading plugin 'docling_defaults'
2026-01-12 21:22:30,346 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-01-12 21:22:30,350 - INFO - Loading plugin 'docling_defaults'
2026-01-12 21:22:30,353 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-01-12 21:22:31,669 - INFO - Auto OCR model selected ocrmac.
2026-01-12 21:22:31,673 - INFO - Loading plugin 'docling_defaults'
2026-01-12 21:22:31,675 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-01-12 21:22:33,394 - INFO - Accelerator device: 'mps'
2026-01-12 21:22:35,194 - INFO - Loading plugin 'docling_defaul

In [11]:
mrk_down = documents.document.export_to_markdown()

In [ ]:
import tiktoken 
enc = tiktoken.get_encoding("o200k_base")

ValueError: Unknown encoding gpt-5-mini.
Plugins found: ['tiktoken_ext.openai_public']
tiktoken version: 0.12.0 (are you on latest?)

In [ ]:
with open('../docs/misc/mrkdown_docling.md', 'w', encoding='utf-8') as f:
    f.write(mrk_down)

In [14]:
categories = [
    "Social and Equity",
    "Environmental and Resources",
    "Governance and Policy",
    "Housing and Infrastructure"
]#unique ones
class PolicyCategory(dspy.Signature):
    """Defines the unique policy categories."""
    category: str = dspy.InputField(choices=categories)  # Restrict input to these categories

In [35]:
#attempt 2
class ExtractAdaptationPolicies(dspy.Signature):
    """
    Task: Extract ALL  policies related to the policy_type defined by the policy_definition from the document.
    
    For each policy found, provide:
      1. A specific, descriptive policy name (not generic headings)
      2. Verbatim snippets (2-4 quotes) from the document that describe the policy
    
    Return two parallel lists of the same length:
      - policy_names: concise but encompassing names for each policy
      - policy_details: for each policy, provide verbatim snippets (word-for-word quotes)
                       from the document. You may use multiple non-contiguous snippets.
      - MAKE SURE THE LISTS ARE ALIGNED BY INDEX AND OF THE SAME LENGTH 
    
    Rules:
      - Policy names should be specific and action-oriented, leave details to the details section and let title be a title
      - Details must be quoted verbatim from the source (no paraphrasing)
      - Lists must be the SAME length and aligned by index
      - Extract ALL policies, even if there are many
    """
    
    # Inputs
    document: Attachments = dspy.InputField()
    country: str = dspy.InputField()
    state_or_province: str | None = dspy.InputField()
    city: str | None = dspy.InputField()
    policy_type: str = dspy.InputField()
    policy_definition: str = dspy.InputField()
    
    # Outputs
    policy_names: list[str] = dspy.OutputField(
        desc="List of specific, descriptive policy names"
    )
    #got it to print out the same result
    policy_details: list[list[str]] = dspy.OutputField(
        desc="List of verbatim snippets for each policy (same length as policy_names for outer list)"
    )
    policy_llm_reasoning: list[str] = dspy.OutputField(
        desc="LLM reasoning for policy extraction. Why was this policy included as a seperate policy from the other policies, what made it unique?"
    )

In [38]:
from dataclasses import dataclass, field
from typing import List
@dataclass
class PDFPolicies:
    """All adaptation policies extracted from a single PDF."""
    pdf_path: str
    country: str
    state_or_province: str | None
    city: str | None
    policy_names: List[str] = field(default_factory=list)
    policy_details: List[List[str]] = field(default_factory=list)  # list of snippets per policy
    policy_llm_reasoning: List[str] = field(default_factory=list)

    def flatten_in_place(self, sep: str = " "):
        """
        Replace policy_details (list[list[str]]) with a single-string list representation.
        After this, policy_details becomes List[List[str]] where each inner list has one joined string.
        """
        self.policy_details = [sep.join(snips) for snips in self.policy_details]
    def add_from_dspy(self, result):
        """Populate from a DSPy result (ExtractAdaptationPolicies signature)."""
        self.policy_names = result.policy_names
        self.policy_details = result.policy_details
        self.policy_llm_reasoning = result.policy_llm_reasoning
        self.flatten_in_place()



In [ ]:
key_to_pdf = {
"Seattle":"../pdfs/Seattle.pdf",

"Las Vegas":"../pdfs/CLV.pdf",
"Miami-Dade":"../pdfs/MIAMI_DADE.pdf",
"Hiroshima":"../pdfs/HIROSHIMA.pdf",
"Geneva":"../pdfs/city_of_geneva.pdf"
}

keys_to_details ={
    "Seattle":['United States','Washington'],
    "Las Vegas":['United States','Nevada'],
    "Miami-Dade":["United States","Florida"],
    "Hiroshima":["Japan","N/A"],
    "Geneva":["Switzerland","N/A"]
}



In [40]:
def build_policy_details(key_to_pdf,keys_to_details):
    city_to_details={}
    for city in key_to_pdf.keys():
        pdf_path = key_to_pdf[city]
        curr_city_policies = PDFPolicies(
        pdf_path=pdf_path,
        country=keys_to_details[city][0],
        state_or_province=keys_to_details[city][1],
        city=city,
        )
        curr_city_results = dspy.ChainOfThought(ExtractAdaptationPolicies)(
        document=Attachments(key_to_pdf[city]),
        country= keys_to_details[city][0],
        state_or_province=keys_to_details[city][1],
        city=city,
        policy_type="adaptation",  # fixed spelling
        policy_definition=ADAPTATION_POLICY_DESCRIPTION, # fixed name
        policy_categories=categories
)
        curr_city_policies.add_from_dspy(curr_city_results)
        city_to_details[city] = curr_city_policies
    return city_to_details
        


In [41]:
built_policies = build_policy_details(key_to_pdf,keys_to_details)


[Attachments] Running primary processor 'pdf_to_llm' for pdfs/Seattle.pdf
[Attachments]   Applying step 'load.url_to_response' to pdfs/Seattle.pdf
[Attachments]   Applying step 'modify.morph_to_detected_type' to pdfs/Seattle.pdf
[Attachments]   Applying step 'load.pdf_to_pdfplumber' to pdfs/Seattle.pdf
[Attachments]   Applying step 'modify.pages' to pdfs/Seattle.pdf
[Attachments]   Running AdditivePipeline(present.markdown + present.images + present.metadata)
[Attachments]     Applying additive step 'present.markdown' to pdfs/Seattle.pdf
[Attachments]     Applying additive step 'present.images' to pdfs/Seattle.pdf
[Attachments]     Applying additive step 'present.metadata' to pdfs/Seattle.pdf
[Attachments]   Applying step 'refine.tile_images' to pdfs/Seattle.pdf
[Attachments]   Applying step 'refine.resize_images' to pdfs/Seattle.pdf
[Attachments] Running primary processor 'pdf_to_llm' for pdfs/CLV.pdf
[Attachments]   Applying step 'load.url_to_response' to pdfs/CLV.pdf
[Attachments]  

In [42]:
class ClassifyPolicy(dspy.Signature):
    """
    Task: Classify a single policy into ONE primary category.
    
    Categories and their definitions:
    
    1. Housing and Infrastructure
       - Physical buildings, construction standards, urban development
       - Transportation networks (roads, transit, cycling infrastructure)
       - Utilities and service infrastructure (water, electricity)
       - Building codes, retrofitting, climate-resilient design
    
    2. Environmental and Resources
       - Natural ecosystems, biodiversity, habitat conservation
       - Water resource management, watershed management
       - Energy resources (renewable energy, energy mix)
       - Air quality, pollution control, environmental monitoring
       - Land use, soil management, forestry
    
    3. Economic and Employment
       - Job creation, workforce development, skills training
       - Business support, economic incentives, financial programs
       - Industry transformation, economic diversification
       - Carbon pricing, green finance, investment frameworks
       - Livelihood support, economic resilience
    
    4. Social and Equity
       - Vulnerable populations, community welfare, social justice
       - Public health, healthcare access, disease prevention
       - Housing affordability, social housing programs
       - Education, awareness campaigns, community engagement
       - Food security, access to services
       - Cultural preservation, indigenous rights
    
    5. Governance and Policy
       - Legislation, regulations, legal frameworks
       - Planning processes, strategic frameworks, policy integration
       - Institutional coordination, inter-agency collaboration
       - Monitoring and evaluation systems, reporting requirements
       - Emergency management, disaster preparedness systems
       - Data management, information systems
    
    Classification Priority:
    1. The explicit stated goal of the policy
    2. The primary action or intervention described
    3. The main beneficiary or target of the policy
    """
    
    policy_name: str = dspy.InputField()
    policy_details: str = dspy.InputField()
    country: str = dspy.InputField()
    state_or_province: str | None = dspy.InputField()
    city: str | None = dspy.InputField()
    
    primary_category: str = dspy.OutputField(
        desc="ONE of: Housing and Infrastructure, Environmental and Resources, "
             "Economic and Employment, Social and Equity, Governance and Policy, NONE"
    )
    confidence: str = dspy.OutputField(
        desc="high, medium, or low"
    )
    reasoning: str = dspy.OutputField(
        desc="2-3 sentences explaining why this policy was created in reference to the other policies that you have already made. Justify why this is a seperate policy from the others, as the LLM why are you identifying this as a separate policy from the others. If NONE explain why you think this policy does not fit into any of the other categories."
    )

In [43]:
def classify_policies(built_polcies):
    #gonna be a city: [{policy_name : result}]
    policy_classifier_dict = {}
    for city in built_polcies.keys():
        city_policy_classifier_list = []
        for i in range(len(built_policies[city].policy_names)):
            policy_name = built_policies[city].policy_names[i]
            policy_details = built_polcies[city].policy_details[i]
            classification_result = dspy.ChainOfThought(ClassifyPolicy)(
                policy_name=policy_name,
                policy_details=policy_details,
                country=built_policies[city].country,
                state_or_province=built_policies[city].state_or_province,
                city=built_policies[city].city,
            )
            city_policy_classifier_list.append({policy_name: classification_result})
        policy_classifier_dict[city] = city_policy_classifier_list
    return policy_classifier_dict



In [46]:
classified_policies = classify_policies(built_policies)


In [47]:
# want to see where the list indices don't match up, if they do, go back and iterate on prompt
#interesting part to do batching for later, since you do need them to match and if you have a big problem then they won't
for city in built_policies.keys():
    num_names = len(built_policies[city].policy_names)
    num_details = len(built_policies[city].policy_details)
    if num_names != num_details:
        print(f"Mismatch in {city}: {num_names} names vs {num_details} details")
        for i in range(max(num_names, num_details)):
            name = built_policies[city].policy_names[i] if i < num_names else None
            detail = built_policies[city].policy_details[i] if i < num_details else None
            print(f"Index {i}: Name = {name}, Detail = {detail}")



In [ ]:
classified_policies

In [49]:
#for each city write down the policy name and policy details, match with built_policies dict per city per policy
master_dict = {}
for city in classified_policies.keys():
    master_dict[city] = []
    for policy_dict in classified_policies[city]:
        for policy_name, classification in policy_dict.items():
            # Find index of policy_name in built_policies to get details
            try:
                index = built_policies[city].policy_names.index(policy_name)
                policy_details = built_policies[city].policy_details[index]
                policy_extraction_reasoning = built_policies[city].policy_llm_reasoning[index]
            except ValueError:
                policy_details = "Details not found"
                policy_extraction_reasoning = "Reasoning not found"
            master_dict[city].append({
                "Policy Name": policy_name,
                "Policy Details": policy_details,
                "Classification": classification.primary_category,
                "Confidence": classification.confidence,
                "Extraction Reasoning": policy_extraction_reasoning,
                "Classification Reasoning": classification.reasoning
            })

In [ ]:
#want to write this to csv 
import csv
with open('../data/policies/classified_policies.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Write header
    writer.writerow(["City", "Policy Name", "Policy Details", "Classification", "Confidence", "Extraction Reasoning", "Classification Reasoning"])
    
    for city, policies in master_dict.items():
        for policy in policies:
            writer.writerow([
                city,
                policy["Policy Name"],
                policy["Policy Details"],
                policy["Classification"],
                policy["Confidence"],
                policy["Extraction Reasoning"],
                policy["Classification Reasoning"]
            ])